In [2]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("e commerce.csv")

# Quick checks
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing % (descending):")
print((df.isna().sum() / len(df) * 100).sort_values(ascending=False).round(2))
print("\nDuplicate rows:", df.duplicated().sum())


Shape: (1666, 22)

Columns: ['id', 'slug', 'title', 'imgs', 'brand', 'category', 'vendor', 'used', 'address', 'availability', 'currency', 'original_price', 'discounted_price', 'specifications', 'description', 'delivery_fee', 'delivery_details', 'warranty', 'warranty_type', 'average_rating', 'num_ratings', 'reviews']

Missing % (descending):
delivery_details    100.00
delivery_fee         93.58
description          82.05
warranty_type        76.65
warranty             69.99
address              67.35
discounted_price     55.64
brand                53.60
num_ratings          45.98
average_rating       45.98
availability         35.23
original_price        1.02
id                    0.00
slug                  0.00
category              0.00
imgs                  0.00
title                 0.00
vendor                0.00
specifications        0.00
currency              0.00
used                  0.00
reviews               0.00
dtype: float64

Duplicate rows: 0


In [3]:
df.drop(columns=['delivery_details'], inplace=True, errors='ignore')


In [4]:
# Convert price columns to numeric (remove currency symbols if present)
df['original_price'] = pd.to_numeric(df['original_price'], errors='coerce')
if 'discounted_price' in df.columns:
    df['discounted_price'] = pd.to_numeric(df['discounted_price'], errors='coerce')

# Ratings -> numeric
if 'average_rating' in df.columns:
    df['average_rating'] = pd.to_numeric(df['average_rating'], errors='coerce')
if 'num_ratings' in df.columns:
    df['num_ratings'] = pd.to_numeric(df['num_ratings'], errors='coerce')

print(df[['original_price','discounted_price','average_rating','num_ratings']].describe())


       original_price  discounted_price  average_rating  num_ratings
count    1.649000e+03        739.000000      900.000000   900.000000
mean     1.615732e+05      41247.326116        4.992222    13.331111
std      2.470286e+05      96631.352491        0.096925    43.200123
min      1.199000e+03       1149.000000        3.500000     1.000000
25%      6.000000e+03       3899.500000        5.000000     1.000000
50%      5.199900e+04       6599.000000        5.000000     1.000000
75%      2.420000e+05      29299.000000        5.000000     6.000000
max      4.457999e+06     794999.000000        5.000000   751.000000


In [5]:
# Brand
df['brand'] = df['brand'].fillna("Unknown")

# Description
df['description'] = df['description'].fillna("No description available")

# Discounted price -> assume not discounted if NaN
if 'discounted_price' in df.columns:
    df['discounted_price'] = df['discounted_price'].fillna(df['original_price'])

# Ratings: fill num_ratings NaN with 0, average_rating with dataset mean or 0
if 'num_ratings' in df.columns:
    df['num_ratings'] = df['num_ratings'].fillna(0).astype(int)
if 'average_rating' in df.columns:
    avg_mean = df['average_rating'].mean(skipna=True)
    df['average_rating'] = df['average_rating'].fillna(round(avg_mean, 2))


In [7]:
# Drop rows where original_price is NaN (optional)
df = df[df['original_price'].notna()].copy()

# IQR outlier removal
Q1 = df['original_price'].quantile(0.25)
Q3 = df['original_price'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df = df[(df['original_price'] >= lower) & (df['original_price'] <= upper)].copy()
print("After removing price outliers, shape:", df.shape)


After removing price outliers, shape: (1512, 21)


In [8]:
df['title'] = df['title'].astype(str).str.strip().str.lower()
df['brand'] = df['brand'].astype(str).str.strip().str.lower()
df['category'] = df['category'].astype(str).str.strip().str.lower()
df['vendor'] = df['vendor'].astype(str).str.strip().str.lower()


In [10]:
# Discount percent
df['discount_pct'] = ((df['original_price'] - df['discounted_price']) / df['original_price']).fillna(0)
df['discount_pct'] = df['discount_pct'].clip(lower=0)

# Price bucket (example bins)
bins = [0, 5000, 15000, 30000, 60000, df['original_price'].max()]
labels = ['very_low', 'low', 'medium', 'high', 'very_high']
df['price_bucket'] = pd.cut(df['original_price'], bins=bins, labels=labels, include_lowest=True)


In [11]:
print("Final shape:", df.shape)
print("\nMissing % after cleaning:")
print((df.isna().sum() / len(df) * 100).sort_values(ascending=False).round(2).head(20))

# Save to CSV
df.to_csv("cleaned_ecommerce.csv", index=False)
print("Saved cleaned_ecommerce.csv")


Final shape: (1512, 23)

Missing % after cleaning:
delivery_fee        93.58
warranty_type       79.37
warranty            74.54
address             70.57
availability        32.01
slug                 0.00
id                   0.00
vendor               0.00
category             0.00
brand                0.00
imgs                 0.00
title                0.00
original_price       0.00
used                 0.00
currency             0.00
description          0.00
specifications       0.00
discounted_price     0.00
average_rating       0.00
num_ratings          0.00
dtype: float64
Saved cleaned_ecommerce.csv
